# Gold Layer: Slowly Changing Dimensions (SCD Type 2) & Delta Lake Merge

This notebook implements the Customer Slowly Changing Dimension Type 2 (`dim_customers_scd2`) in the Databricks Lakehouse:
- Ingests cleaned customer records from `workspace.ecompulse_silver.customers`.
- Materializes `workspace.ecompulse_gold.dim_customers_scd2` tracking geographic relocations (`customer_city`, `customer_state`, `customer_zip_code_prefix`).
- Derives SHA-256 surrogate keys (`customer_sk`) concatenating natural business key and effective start date.
- Executes staged atomic Delta Lake `MERGE` (`DeltaTable.forName`) to close expired rows (`is_current = False`) and insert new versions (`is_current = True`).
- Implements watermark-based incremental loading using `max(effective_start_date)` to avoid full table rewrites.
- Demonstrates Delta Lake Schema Enforcement (rejection of incompatible schemas) and Schema Evolution (`.option("mergeSchema", "true")`).
- Demonstrates Delta Lake Time Travel (`versionAsOf`, `timestampAsOf`, `VERSION AS OF`, `TIMESTAMP AS OF`) and transaction log auditing (`DESCRIBE HISTORY`, `DeltaTable.history()`).
- Includes terminal `%sql` verification cells for inspection.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# Unity Catalog target table coordinates
TARGET_TABLE_NAME = "workspace.ecompulse_gold.dim_customers_scd2"

In [0]:
# Create target Delta table if it does not already exist (DDL Bootstrap)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE_NAME} (
    customer_sk STRING NOT NULL,
    customer_id STRING NOT NULL,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    effective_start_date TIMESTAMP NOT NULL,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN NOT NULL
) USING DELTA
COMMENT 'Customer Slowly Changing Dimension Type 2 with SHA-256 Surrogate Keys'
""")

## Section 1: Batch 1 — Baseline Ingestion & Table Initialization

Ingests the baseline customer snapshot from `workspace.ecompulse_silver.customers`.
All records are initialized as currently active (`is_current = True`) with `effective_start_date = '2023-01-01 00:00:00'` and open-ended validity (`effective_end_date = NULL`).

In [0]:
# Batch 1 Baseline Ingestion (Initial Load)
batch_1_timestamp = F.to_timestamp(F.lit("2023-01-01 00:00:00"))

# Read cleaned customer records from Silver layer
silver_customers = spark.read.table("workspace.ecompulse_silver.customers")

# Cleanse, cast data types, deduplicate natural key, and derive validity columns
dim_customers_batch1 = (
    silver_customers
    .select(
        F.col("customer_id"),
        F.col("customer_zip_code_prefix").cast("int").alias("customer_zip_code_prefix"),
        F.col("customer_city"),
        F.col("customer_state")
    )
    .dropDuplicates(["customer_id"])
    .withColumn("effective_start_date", batch_1_timestamp)
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn(
        "customer_sk",
        F.sha2(
            F.concat_ws("_", F.col("customer_id"), F.col("effective_start_date").cast("string")),
            256
        )
    )
    .select(
        "customer_sk",
        "customer_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
)

# Materialize baseline load into Delta table (Version 0)
dim_customers_batch1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TABLE_NAME)

print(f"Batch 1 committed successfully to {TARGET_TABLE_NAME}. Total baseline records: {dim_customers_batch1.count()}")

## Section 2: Batch 2 — Incremental Loading & Watermarking (F18)

In Batch 2, we simulate an incoming incremental feed containing:
1. **Relocated Customers**: Existing customers who moved to new cities/states (requiring SCD2 versioning).
2. **New Customer Sign-ups**: Brand new customer records not present in the baseline.
3. **Unchanged Customers**: Existing customers whose attributes have not changed (no-op).

We inspect the high watermark from `dim_customers_scd2` and process the delta without rewriting the full table.

In [0]:
# Watermark-based incremental load logic
watermark_df = spark.table(TARGET_TABLE_NAME).select(
    F.max("effective_start_date").alias("last_watermark")
)
last_loaded_watermark = watermark_df.collect()[0]["last_watermark"]
print(f"Current Target High Watermark: {last_loaded_watermark}")

# Establish timestamp for Batch 2 incremental load
batch_2_timestamp = F.to_timestamp(F.lit("2023-06-01 00:00:00"))

# Construct incoming incremental batch containing:
# 1. Relocated customer (attribute update -> close out old version, insert new version)
# 2. Unchanged customer (idempotent pass-through -> no-op)
# 3. New sign-ups (insert new version)
active_samples = (
    spark.table(TARGET_TABLE_NAME)
    .filter(F.col("is_current") == True)
    .limit(2)
    .collect()
)

c1_id = active_samples[0]["customer_id"] if len(active_samples) > 0 else "c1"
c2_id = active_samples[1]["customer_id"] if len(active_samples) > 1 else "c2"

batch_2_data = [
    (c1_id, 13000, "Campinas", "SP"),               # Relocated to Campinas
    (c2_id, 20000, "Rio de Janeiro", "RJ"),         # Unchanged
    ("c_new_99901", 88000, "Florianopolis", "SC"),   # New customer 1
    ("c_new_99902", 40000, "Salvador", "BA"),        # New customer 2
]

batch_2_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

batch_2_df = spark.createDataFrame(batch_2_data, schema=batch_2_schema)
print(f"Incoming Batch 2 Delta Records: {batch_2_df.count()}")

## Section 3: Atomic Staged Delta MERGE (F17)

To handle SCD Type 2 updates and inserts within a single atomic Delta transaction, we apply the Staged Union Pattern:
- **Updates**: Incoming rows that match active target rows with changed attributes are staged with `merge_key = customer_id`. Delta MERGE executes `whenMatchedUpdate` to set `is_current = False` and `effective_end_date = batch_2_timestamp`.
- **Inserts**: New versions of changed customers and brand new customers are staged with `merge_key = NULL`. Delta MERGE executes `whenNotMatchedInsert` to insert them as active records with new SHA-256 surrogate keys.

In [0]:
# Staged Delta MERGE: Identify changes and stage updates/inserts
target_delta = DeltaTable.forName(spark, TARGET_TABLE_NAME)
target_df = target_delta.toDF()
target_active = target_df.filter(F.col("is_current") == True)

# Left join incoming delta data with active target records on natural key
diff_df = batch_2_df.alias("src").join(
    target_active.alias("tgt"),
    on="customer_id",
    how="left"
)

# Detect attribute changes across tracked geographic attributes
attribute_changed = (
    (F.col("src.customer_city") != F.col("tgt.customer_city")) |
    (F.col("src.customer_state") != F.col("tgt.customer_state")) |
    (F.col("src.customer_zip_code_prefix") != F.col("tgt.customer_zip_code_prefix")) |
    (F.col("src.customer_city").isNull() != F.col("tgt.customer_city").isNull()) |
    (F.col("src.customer_state").isNull() != F.col("tgt.customer_state").isNull()) |
    (F.col("src.customer_zip_code_prefix").isNull() != F.col("tgt.customer_zip_code_prefix").isNull())
)

# Identify rows to close out and rows to insert
rows_to_update = diff_df.filter(F.col("tgt.customer_id").isNotNull() & attribute_changed)
rows_to_insert = diff_df.filter(F.col("tgt.customer_id").isNull() | attribute_changed)

# Branch 1: Staged Updates (merge_key = customer_id to match active target record)
stage_updates = rows_to_update.select(
    F.col("src.customer_id").alias("merge_key"),
    F.lit(None).cast("string").alias("customer_sk"),
    F.col("src.customer_id").alias("customer_id"),
    F.col("src.customer_zip_code_prefix").alias("customer_zip_code_prefix"),
    F.col("src.customer_city").alias("customer_city"),
    F.col("src.customer_state").alias("customer_state"),
    F.lit(None).cast("timestamp").alias("effective_start_date"),
    batch_2_timestamp.alias("effective_end_date"),
    F.lit(False).alias("is_current")
)

# Branch 2: Staged Inserts (merge_key = NULL so it never matches, triggering whenNotMatchedInsert)
stage_inserts = rows_to_insert.select(
    F.lit(None).cast("string").alias("merge_key"),
    F.sha2(
        F.concat_ws("_", F.col("src.customer_id"), batch_2_timestamp.cast("string")),
        256
    ).alias("customer_sk"),
    F.col("src.customer_id").alias("customer_id"),
    F.col("src.customer_zip_code_prefix").alias("customer_zip_code_prefix"),
    F.col("src.customer_city").alias("customer_city"),
    F.col("src.customer_state").alias("customer_state"),
    batch_2_timestamp.alias("effective_start_date"),
    F.lit(None).cast("timestamp").alias("effective_end_date"),
    F.lit(True).alias("is_current")
)

# Combine both staging branches into unified DataFrame
staged_df = stage_updates.unionByName(stage_inserts)

# Execute Atomic Staged MERGE on Delta Table
(
    target_delta.alias("target")
    .merge(
        source=staged_df.alias("source"),
        condition="target.customer_id = source.merge_key AND target.is_current = true"
    )
    .whenMatchedUpdate(
        set={
            "is_current": F.lit(False),
            "effective_end_date": F.col("source.effective_end_date")
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_sk": F.col("source.customer_sk"),
            "customer_id": F.col("source.customer_id"),
            "customer_zip_code_prefix": F.col("source.customer_zip_code_prefix"),
            "customer_city": F.col("source.customer_city"),
            "customer_state": F.col("source.customer_state"),
            "effective_start_date": F.col("source.effective_start_date"),
            "effective_end_date": F.col("source.effective_end_date"),
            "is_current": F.col("source.is_current")
        }
    )
    .execute()
)

print(f"Delta MERGE committed successfully into {TARGET_TABLE_NAME}.")

In [0]:
# Verify SCD Type 2 state transitions for updated customer
c1_history = (
    spark.table(TARGET_TABLE_NAME)
    .filter(F.col("customer_id") == c1_id)
    .orderBy("effective_start_date")
    .collect()
)

print(f"Customer {c1_id} Version History Count: {len(c1_history)}")
for idx, v in enumerate(c1_history):
    print(f"  Version {idx + 1}: city={v['customer_city']}, is_current={v['is_current']}, start={v['effective_start_date']}, end={v['effective_end_date']}, sk={v['customer_sk'][:16]}...")

## Section 4: Delta Lake Schema Enforcement & Evolution (F19)

Delta Lake protects table integrity at write time:
1. **Schema Enforcement**: Blocks writes that contain undeclared columns.
2. **Schema Evolution**: Allows safe introduction of new columns (`customer_country`, `loyalty_tier`) when explicitly authorized via `.option("mergeSchema", "true")`.

In [0]:
# 1. Demonstrate Delta Lake Schema Enforcement (validation rejects undeclared columns)
mismatched_schema_df = spark.createDataFrame([
    ("cust_enforce_001", 1001, "Sao Paulo", "SP", "Brazil", "VIP_PLATINUM")
], ["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state", "customer_country", "loyalty_tier"])

print("Attempting to write mismatched schema DataFrame without mergeSchema...")
try:
    mismatched_schema_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TARGET_TABLE_NAME)
    print("WARNING: Write unexpectedly succeeded — schema enforcement did not trigger!")
except Exception as e:
    print(f"SUCCESS: Delta Lake Schema Enforcement successfully rejected mismatched write: {type(e).__name__}")

# 2. Demonstrate Delta Lake Schema Evolution via .option("mergeSchema", "true")
batch_timestamp_evolution = F.current_timestamp()

evolved_records_df = spark.createDataFrame([
    ("cust_evo_001", 80000, "Curitiba", "PR", "Brazil", "Gold"),
    ("cust_evo_002", 90000, "Porto Alegre", "RS", "Brazil", "Platinum")
], ["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state", "customer_country", "loyalty_tier"]) \
  .withColumn("effective_start_date", batch_timestamp_evolution) \
  .withColumn("effective_end_date", F.lit(None).cast("timestamp")) \
  .withColumn("is_current", F.lit(True)) \
  .withColumn(
      "customer_sk",
      F.sha2(
          F.concat_ws("_", F.col("customer_id"), F.col("effective_start_date").cast("string")),
          256
      )
  )

print("Applying schema evolution with .option('mergeSchema', 'true')...")
evolved_records_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(TARGET_TABLE_NAME)

print(f"Schema evolution successful! Updated table columns: {spark.table(TARGET_TABLE_NAME).columns}")

## Section 5: Delta Lake Time Travel & Transaction Auditing (F20)

Delta Lake maintains an immutable transaction log (`_delta_log/`), enabling point-in-time time travel queries and auditing via:
- `versionAsOf` and `timestampAsOf` options in PySpark DataFrame reader.
- `DeltaTable.history()` and SQL `DESCRIBE HISTORY`.
- SQL `VERSION AS OF` and `TIMESTAMP AS OF`.

In [0]:
# Delta Lake Time Travel & History (PySpark API)
delta_dim_table = DeltaTable.forName(spark, TARGET_TABLE_NAME)

# 1. Inspect transaction log commit history (auditing)
history_df = delta_dim_table.history()
print("Delta Transaction Log History:")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(5, truncate=False)

# 2. Point-in-time time travel via versionAsOf (Version 0: initial baseline)
dim_v0_df = spark.read.format("delta").option("versionAsOf", 0).table(TARGET_TABLE_NAME)
print(f"Version 0 Baseline Row Count: {dim_v0_df.count()}")

# 3. Point-in-time time travel via timestampAsOf
earliest_commit_timestamp = history_df.orderBy("version").select("timestamp").first()[0]
earliest_ts_str = str(earliest_commit_timestamp)
dim_ts_df = spark.read.format("delta").option("timestampAsOf", earliest_ts_str).table(TARGET_TABLE_NAME)
print(f"Timestamp As Of ('{earliest_ts_str}') Row Count: {dim_ts_df.count()}")

In [0]:
%sql
-- Audit transaction history of dim_customers_scd2
DESCRIBE HISTORY workspace.ecompulse_gold.dim_customers_scd2;

In [0]:
%sql
-- Time travel query: Historical state at Version 0
SELECT customer_sk, customer_id, customer_city, customer_state, customer_zip_code_prefix, effective_start_date, effective_end_date, is_current
FROM workspace.ecompulse_gold.dim_customers_scd2 VERSION AS OF 0
LIMIT 10;

In [0]:
%sql
-- Time travel query: Historical state at current_timestamp()
SELECT customer_sk, customer_id, customer_city, customer_state, is_current, effective_start_date
FROM workspace.ecompulse_gold.dim_customers_scd2 TIMESTAMP AS OF current_timestamp()
LIMIT 10;

In [0]:
%sql
-- Terminal Verification: Active vs Expired Dimension Records
SELECT 
    customer_sk,
    customer_id,
    customer_city,
    customer_state,
    customer_zip_code_prefix,
    effective_start_date,
    effective_end_date,
    is_current
FROM workspace.ecompulse_gold.dim_customers_scd2
ORDER BY customer_id ASC, effective_start_date ASC
LIMIT 20;

In [0]:
%sql
-- Terminal Verification: Active and Expired Record Counts
SELECT 
    is_current,
    count(1) AS total_records,
    min(effective_start_date) AS earliest_start,
    max(effective_start_date) AS latest_start
FROM workspace.ecompulse_gold.dim_customers_scd2
GROUP BY is_current;